## Cell 1: Setup
**What this demonstrates**: Graph RAG requires a graph library (networkx) alongside the standard RAG stack.
**Expected output**: All imports succeed; API clients initialised; constants printed.

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
# Uncomment on first run:
# !pip install anthropic openai langchain-openai langchain-community chromadb \
#              networkx python-dotenv --quiet

# ── Standard library ─────────────────────────────────────────────────────────
from __future__ import annotations
import json
import os
import re
from collections import defaultdict, deque
from dataclasses import dataclass, field
from typing import Any

# ── Third-party ──────────────────────────────────────────────────────────────
import networkx as nx
from anthropic import Anthropic
from dotenv import load_dotenv
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

load_dotenv()

# ── API clients ───────────────────────────────────────────────────────────────
client     = Anthropic()   # reads ANTHROPIC_API_KEY
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')  # reads OPENAI_API_KEY

# ── Constants ─────────────────────────────────────────────────────────────────
HAIKU   = 'claude-haiku-4-5-20251001'   # entity extraction (high volume, cheap)
SONNET  = 'claude-sonnet-4-6'           # final synthesis
MAX_HOPS        = 2    # BFS depth for graph traversal
MAX_GRAPH_NODES = 20   # cap on nodes returned per traversal
TOP_K_VECTOR    = 4    # chunks from vector search

print('Graph RAG — Module 24')
print(f'  Reasoning model  : {SONNET}')
print(f'  Extraction model : {HAIKU}')
print(f'  Max traversal hops   : {MAX_HOPS}')
print(f'  Max graph nodes  : {MAX_GRAPH_NODES}')
print(f'  Vector top-k     : {TOP_K_VECTOR}')
print('\nLibraries loaded: networkx, anthropic, langchain-openai, chromadb')


## Cell 2: Data — Synthetic Financial News Corpus
**What this demonstrates**: A corpus of short financial documents whose entities
and relationships are spread across multiple files. No single document contains
the full picture — the graph assembles it.
**Expected output**: 8 documents printed with word counts; entity statistics.

In [ ]:
# Synthetic financial news + regulatory corpus.
# Documents are deliberately short so the full corpus is readable in the demo.
# Entities span all docs; relationships only emerge when the graph is assembled.

CORPUS: list[dict[str, str]] = [
    {
        'id': 'doc_01',
        'title': 'Lehman Brothers Files for Bankruptcy',
        'text': (
            'Lehman Brothers Holdings Inc. filed for Chapter 11 bankruptcy protection '
            'on 15 September 2008, marking the largest bankruptcy in US history with '
            '$639 billion in assets. Barclays Capital subsequently acquired Lehman\'s '
            'North American investment banking and capital markets businesses. '
            'Nomura Holdings acquired Lehman\'s Asia-Pacific and European operations.'
        ),
    },
    {
        'id': 'doc_02',
        'title': 'Reserve Primary Fund Breaks the Buck',
        'text': (
            'The Reserve Primary Fund, a money market fund managed by Reserve Management '
            'Company, held $785 million in Lehman Brothers commercial paper. Following the '
            'Lehman bankruptcy, the fund\'s net asset value fell below $1 per share — '
            'a condition known as \'breaking the buck\'. This triggered mass redemption '
            'requests across the money market sector.'
        ),
    },
    {
        'id': 'doc_03',
        'title': 'AIG Counterparty Exposure Report',
        'text': (
            'AIG Financial Products had written credit default swaps (CDS) referencing '
            'Lehman Brothers obligations with a notional value of approximately $22 billion. '
            'Goldman Sachs held $20 billion in CDS protection purchased from AIG. '
            'Morgan Stanley held $11 billion in AIG-written CDS. The US Federal Reserve '
            'extended an $85 billion emergency credit facility to AIG to prevent systemic '
            'contagion to CDS counterparties.'
        ),
    },
    {
        'id': 'doc_04',
        'title': 'Basel III Capital Requirements for Counterparty Risk',
        'text': (
            'Under Basel III, banks must hold additional capital against counterparty '
            'credit risk (CCR) exposures. Goldman Sachs disclosed peak CCR exposure '
            'of $34 billion during the 2008 crisis. Morgan Stanley reported CCR '
            'exposure of $28 billion. Both institutions subsequently raised Tier 1 '
            'capital ratios to comply with Basel III minimum requirements of 6 percent.'
        ),
    },
    {
        'id': 'doc_05',
        'title': 'Barclays Post-Acquisition Integration',
        'text': (
            'Following the acquisition of Lehman Brothers\' North American business, '
            'Barclays Capital integrated approximately 10,000 Lehman employees. '
            'Barclays assumed $47.4 billion in Lehman securities alongside '
            '$45.5 billion in liabilities. The Federal Reserve Bank of New York '
            'facilitated the transaction to ensure orderly market function.'
        ),
    },
    {
        'id': 'doc_06',
        'title': 'Money Market Contagion and Fed Intervention',
        'text': (
            'The collapse of the Reserve Primary Fund prompted the US Treasury to '
            'guarantee money market fund balances above $1 per share. The Federal '
            'Reserve established the Money Market Investor Funding Facility (MMIFF) '
            'to provide liquidity. Fidelity Investments and Vanguard reported '
            'significant inflows as investors sought government-backed alternatives.'
        ),
    },
    {
        'id': 'doc_07',
        'title': 'Goldman Sachs Crisis Management',
        'text': (
            'Goldman Sachs received $10 billion in TARP funding from the US Treasury '
            'in October 2008. The firm had $13 billion in net exposure to AIG at the '
            'time of the AIG bailout. Goldman Sachs repaid all TARP funds in June 2009. '
            'The firm converted to a bank holding company regulated by the Federal Reserve.'
        ),
    },
    {
        'id': 'doc_08',
        'title': 'Regulatory Response and Dodd-Frank',
        'text': (
            'The Dodd-Frank Wall Street Reform Act of 2010 introduced mandatory central '
            'clearing for OTC derivatives, including credit default swaps. Goldman Sachs, '
            'Morgan Stanley, and JPMorgan Chase were designated as systemically important '
            'financial institutions (SIFIs) subject to enhanced prudential standards. '
            'The Financial Stability Oversight Council (FSOC) was established to monitor '
            'systemic risk across interconnected financial entities.'
        ),
    },
]

# Display corpus statistics
total_words = 0
print('Corpus documents:')
print('-' * 60)
for doc in CORPUS:
    words = len(doc['text'].split())
    total_words += words
    print(f"  [{doc['id']}] {doc['title']}")
    print(f'           {words} words')

print('-' * 60)
print(f'Total: {len(CORPUS)} documents, {total_words} words')
print()
print('Key entities present across corpus:')
print('  Organisations : Lehman Brothers, AIG, Goldman Sachs, Morgan Stanley,')
print('                  Barclays Capital, Nomura Holdings, Reserve Primary Fund,')
print('                  Federal Reserve, US Treasury, JPMorgan Chase')
print('  Instruments   : CDS, commercial paper, TARP, Tier 1 capital')
print('  Regulations   : Basel III, Dodd-Frank, SIFI designation')


## Cell 3: Core — Entity Extraction, Graph Construction, and Hybrid Retrieval
**What this demonstrates**: (1) Extract entity triples from each document using
Claude Haiku. (2) Build a NetworkX directed graph from those triples. (3) Implement
BFS-based local traversal. (4) Detect themes via connected-component community detection.
(5) Combine graph context with ChromaDB vector search.
**Expected output**: Graph built (node/edge counts); vector index created; functions ready.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Part A: Entity and relationship extraction
# ─────────────────────────────────────────────────────────────────────────────

EXTRACTION_PROMPT = '''\
Extract entities and relationships from this financial text.

Return a JSON object with two keys:
- "entities": list of {{"name": str, "type": str}}
  Types: ORGANISATION, PERSON, REGULATION, INSTRUMENT, EVENT, AGENCY
- "triples": list of {{"subject": str, "predicate": str, "object": str}}
  Predicates (use exactly): OWNS, ACQUIRED, EXPOSED_TO, HOLDS_CDS_ON,
  REGULATED_BY, RECEIVED_FUNDING_FROM, PROVIDED_FUNDING_TO, TRIGGERED,
  SUBSIDIARY_OF, COMPETITOR_OF, DESIGNATED_AS, CLEARED_THROUGH

Rules:
- Normalise entity names: remove legal suffixes (Inc., Ltd., Holdings),
  use canonical names (e.g. "Federal Reserve" not "US Federal Reserve")
- Only extract relationships explicitly stated in the text
- Return valid JSON only, no prose

Text:
{text}'''


def extract_entities_and_triples(doc: dict[str, str]) -> dict[str, Any]:
    '''Call Haiku to extract entity triples from one document.

    Returns the raw JSON dict from the model, plus the source doc_id.
    '''
    response = client.messages.create(
        model=HAIKU,
        max_tokens=1024,
        messages=[{
            'role': 'user',
            'content': EXTRACTION_PROMPT.format(text=doc['text']),
        }],
    )
    raw = response.content[0].text.strip()
    # Strip markdown code fences if the model adds them
    raw = re.sub(r'^```(?:json)?\n?', '', raw)
    raw = re.sub(r'\n?```$', '', raw)
    result = json.loads(raw)
    result['doc_id'] = doc['id']
    return result


# Extract from all documents (sequential; for production use asyncio)
print('Extracting entities from corpus...')
all_extractions: list[dict[str, Any]] = []
for doc in CORPUS:
    extraction = extract_entities_and_triples(doc)
    all_extractions.append(extraction)
    n_entities = len(extraction.get('entities', []))
    n_triples  = len(extraction.get('triples', []))
    print(f"  [{doc['id']}] {n_entities} entities, {n_triples} triples")

print()


# ─────────────────────────────────────────────────────────────────────────────
# Part B: Build the NetworkX knowledge graph
# ─────────────────────────────────────────────────────────────────────────────

def normalise(name: str) -> str:
    '''Lowercase + strip common legal suffixes for entity deduplication.'''
    suffixes = [
        ' inc', ' inc.', ' ltd', ' ltd.', ' llc', ' plc',
        ' holdings', ' corporation', ' corp', ' group',
    ]
    n = name.lower().strip()
    for sfx in suffixes:
        if n.endswith(sfx):
            n = n[: -len(sfx)].strip()
    return n


def build_graph(extractions: list[dict[str, Any]]) -> nx.DiGraph:
    '''Accumulate extracted triples into a directed knowledge graph.

    Each edge stores the predicate label, the source doc_id list (provenance),
    and a human-readable description.
    '''
    G = nx.DiGraph()

    # Track entity types for node attributes
    entity_types: dict[str, str] = {}
    for ext in extractions:
        for ent in ext.get('entities', []):
            canon = normalise(ent['name'])
            entity_types[canon] = ent.get('type', 'UNKNOWN')

    # Add nodes with type attribute
    for canon, etype in entity_types.items():
        G.add_node(canon, entity_type=etype, display_name=canon)

    # Add edges from triples; merge duplicates by extending source_docs list
    for ext in extractions:
        doc_id = ext['doc_id']
        for triple in ext.get('triples', []):
            subj = normalise(triple.get('subject', ''))
            pred = triple.get('predicate', '')
            obj  = normalise(triple.get('object', ''))
            if not subj or not obj or not pred:
                continue
            # Ensure nodes exist even if not in entity list
            if subj not in G:
                G.add_node(subj, entity_type='UNKNOWN', display_name=subj)
            if obj not in G:
                G.add_node(obj, entity_type='UNKNOWN', display_name=obj)
            # Merge or create edge
            if G.has_edge(subj, obj):
                existing = G[subj][obj]
                if doc_id not in existing['source_docs']:
                    existing['source_docs'].append(doc_id)
                if pred not in existing['predicates']:
                    existing['predicates'].append(pred)
            else:
                G.add_edge(subj, obj, predicates=[pred], source_docs=[doc_id])

    return G


KG = build_graph(all_extractions)

print('Knowledge graph built:')
print(f'  Nodes (entities) : {KG.number_of_nodes()}')
print(f'  Edges (relations): {KG.number_of_edges()}')
print()
print('Sample triples (first 8 edges):')
for i, (u, v, data) in enumerate(KG.edges(data=True)):
    if i >= 8:
        break
    preds = ', '.join(data['predicates'])
    docs  = ', '.join(data['source_docs'])
    print(f'  {u} --[{preds}]--> {v}  (source: {docs})')


# ─────────────────────────────────────────────────────────────────────────────
# Part C: Graph traversal — BFS neighbourhood
# ─────────────────────────────────────────────────────────────────────────────

def traverse_graph(
    G: nx.DiGraph,
    query_entities: list[str],
    max_hops: int = MAX_HOPS,
    max_nodes: int = MAX_GRAPH_NODES,
) -> dict[str, Any]:
    '''BFS from each query entity up to max_hops, collecting nodes and edges.

    Returns a dict with nodes, edges, and a relationship_summary string
    ready to be inserted into the synthesis prompt.
    '''
    seeds = [normalise(e) for e in query_entities]
    seeds = [s for s in seeds if s in G]

    visited_nodes: set[str] = set(seeds)
    visited_edges: list[tuple[str, str, str]] = []
    queue: deque[tuple[str, int]] = deque((s, 0) for s in seeds)

    while queue and len(visited_nodes) < max_nodes:
        node, depth = queue.popleft()
        if depth >= max_hops:
            continue
        # Outgoing edges
        for _, neighbour, data in G.out_edges(node, data=True):
            for pred in data['predicates']:
                visited_edges.append((node, pred, neighbour))
            if neighbour not in visited_nodes:
                visited_nodes.add(neighbour)
                queue.append((neighbour, depth + 1))
        # Incoming edges (relationships TO the entity also matter)
        for predecessor, _, data in G.in_edges(node, data=True):
            for pred in data['predicates']:
                visited_edges.append((predecessor, pred, node))
            if predecessor not in visited_nodes:
                visited_nodes.add(predecessor)
                queue.append((predecessor, depth + 1))

    lines = ['Entity relationships found in knowledge graph:']
    for subj, pred, obj in visited_edges:
        subj_type = G.nodes[subj].get('entity_type', '?') if subj in G else '?'
        obj_type  = G.nodes[obj].get('entity_type', '?') if obj in G else '?'
        lines.append(f'  {subj} ({subj_type}) --[{pred}]--> {obj} ({obj_type})')

    return {
        'seeds': seeds,
        'nodes': visited_nodes,
        'edges': visited_edges,
        'relationship_summary': '\n'.join(lines),
    }


# ─────────────────────────────────────────────────────────────────────────────
# Part D: Community detection for thematic summaries
# ─────────────────────────────────────────────────────────────────────────────

def detect_communities(G: nx.DiGraph) -> list[list[str]]:
    '''Find weakly connected components as a simple community proxy.

    For production use, replace with Leiden or Louvain on the undirected
    projection. Connected components identify thematic clusters in small graphs.
    '''
    undirected = G.to_undirected()
    components = list(nx.connected_components(undirected))
    return sorted([list(c) for c in components], key=len, reverse=True)


communities = detect_communities(KG)
print()
print(f'Communities detected (weakly connected components): {len(communities)}')
for i, community in enumerate(communities):
    preview = ', '.join(sorted(community)[:6])
    suffix  = ' ...' if len(community) > 6 else ''
    print(f'  Community {i+1} ({len(community)} nodes): {preview}{suffix}')


# ─────────────────────────────────────────────────────────────────────────────
# Part E: Vector index
# ─────────────────────────────────────────────────────────────────────────────

def build_vector_index(corpus: list[dict[str, str]]) -> Chroma:
    '''Embed each document as a single chunk and store in ChromaDB.

    Title is prepended so the embedding captures the topic as well as content.
    '''
    texts = [f"{doc['title']}\n\n{doc['text']}" for doc in corpus]
    ids   = [doc['id'] for doc in corpus]
    return Chroma.from_texts(
        texts=texts,
        embedding=embeddings,
        ids=ids,
        collection_name='graph_rag_demo',
    )


VECTOR_INDEX = build_vector_index(CORPUS)
print()
print(f'Vector index built: {len(CORPUS)} documents indexed')


# ─────────────────────────────────────────────────────────────────────────────
# Part F: Hybrid retrieval — graph traversal + vector search
# ─────────────────────────────────────────────────────────────────────────────

def extract_query_entities(query: str) -> list[str]:
    '''Ask Haiku to extract named entities from the user query.'''
    response = client.messages.create(
        model=HAIKU,
        max_tokens=256,
        messages=[{
            'role': 'user',
            'content': (
                'List the named entities (organisations, people, regulations) '
                'in this query. Return a JSON array of strings only.\n\n'
                f'Query: {query}'
            ),
        }],
    )
    raw = response.content[0].text.strip()
    raw = re.sub(r'^```(?:json)?\n?', '', raw)
    raw = re.sub(r'\n?```$', '', raw)
    return json.loads(raw)


def graph_rag_retrieve(
    query: str,
    G: nx.DiGraph,
    vector_index: Chroma,
    top_k: int = TOP_K_VECTOR,
) -> dict[str, Any]:
    '''Full hybrid retrieval: graph traversal + vector search.

    Returns graph context, vector chunks, and a merged context string
    ready for the synthesis prompt.
    '''
    query_entities = extract_query_entities(query)
    graph_result   = traverse_graph(G, query_entities)

    vector_docs = vector_index.similarity_search_with_score(query, k=top_k)
    vector_chunks = [
        {'text': doc.page_content, 'score': float(score), 'id': doc.metadata.get('id', '?')}
        for doc, score in vector_docs
    ]

    # Graph relationship summary first — structural scaffold for the LLM
    passage_block = '\n\n---\n\n'.join(
        f"[{c['id']}] {c['text']}" for c in vector_chunks
    )
    merged_context = (
        graph_result['relationship_summary']
        + '\n\n## Retrieved Passages\n\n'
        + passage_block
    )

    return {
        'query_entities': query_entities,
        'graph_nodes_reached': len(graph_result['nodes']),
        'graph_edges_found': len(graph_result['edges']),
        'graph_context': graph_result['relationship_summary'],
        'vector_chunks': vector_chunks,
        'merged_context': merged_context,
    }


print()
print('All components ready:')
print('  extract_entities_and_triples()  — Haiku extraction per document')
print('  build_graph()                   — NetworkX DiGraph builder')
print('  traverse_graph()                — BFS neighbourhood traversal')
print('  detect_communities()            — weakly connected components')
print('  graph_rag_retrieve()            — hybrid graph + vector retrieval')


## Cell 4: Run — End-to-End Graph RAG Query
**What this demonstrates**: Full pipeline: query entity extraction → graph traversal →
vector search → merged context → Claude Sonnet synthesis.
**Expected output**: Graph retrieval statistics; synthesised answer with entity provenance.

In [ ]:
# The Lehman Brothers network query requires traversing multiple entity hops.
# Vector-only search finds chunks mentioning Lehman; the graph finds the full
# network of counterparties and their relationship types across all documents.

QUERY = (
    'What entities are connected to Lehman Brothers and what were the downstream '
    'exposures? Include the instruments involved and the regulatory response.'
)

print(f'Query: {QUERY}')
print('=' * 70)

# ── Retrieve ──────────────────────────────────────────────────────────────────
retrieval = graph_rag_retrieve(QUERY, KG, VECTOR_INDEX)

print(f'\nQuery entities identified : {retrieval["query_entities"]}')
print(f'Graph nodes reached       : {retrieval["graph_nodes_reached"]}')
print(f'Graph edges found         : {retrieval["graph_edges_found"]}')
print(f'Vector chunks retrieved   : {len(retrieval["vector_chunks"])}')

print('\n--- Graph context (entity relationships) ---')
print(retrieval['graph_context'])

print('\n--- Vector passages (top 2 shown) ---')
for chunk in retrieval['vector_chunks'][:2]:
    preview = chunk['text'][:200].replace('\n', ' ')
    print(f"  [{chunk['id']}] score={chunk['score']:.3f}  {preview}...")

# ── Synthesise ────────────────────────────────────────────────────────────────
SYSTEM = (
    'You are a financial analyst. Answer the question using ONLY the provided '
    'context. Structure your answer as follows:\n'
    '1. Direct connections to the queried entity (cite relationship types)\n'
    '2. Downstream/indirect exposures (cite hop-2 relationships)\n'
    '3. Instruments and amounts involved (cite source documents)\n'
    '4. Regulatory response triggered\n'
    'Cite entity relationships from the graph context and passages from the '
    'retrieved text. Do not invent facts not present in the context.'
)

response = client.messages.create(
    model=SONNET,
    max_tokens=800,
    system=SYSTEM,
    messages=[{
        'role': 'user',
        'content': (
            f'Question: {QUERY}\n\n'
            f'Context:\n{retrieval["merged_context"]}'
        ),
    }],
)

answer = response.content[0].text
print('\n' + '=' * 70)
print('GRAPH RAG ANSWER')
print('=' * 70)
print(answer)


## Cell 5: Inspect — Graph Structure, Traversal Paths, and Baseline Comparison
**What this demonstrates**: (1) Graph topology — degree distribution, most-connected
entities. (2) BFS traversal layers showing hop-by-hop discovery. (3) Vector-only
baseline vs graph-augmented answer — the comparison shows what graph context adds.
**Expected output**: Degree table; hop-layer diagram; side-by-side comparison.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Part A: Graph topology — most connected entities (systemic hubs)
# ─────────────────────────────────────────────────────────────────────────────
print('== Graph topology ==')
print(f'Nodes : {KG.number_of_nodes()}')
print(f'Edges : {KG.number_of_edges()}')
print()

degrees = sorted(
    [(n, KG.degree(n), KG.nodes[n].get('entity_type', '?')) for n in KG.nodes()],
    key=lambda x: x[1],
    reverse=True,
)
print('Top 10 entities by total degree (in + out):')
print(f"  {'Entity':<35} {'Type':<16} {'Degree':>6}")
print('  ' + '-' * 60)
for name, deg, etype in degrees[:10]:
    print(f'  {name:<35} {etype:<16} {deg:>6}')

# ─────────────────────────────────────────────────────────────────────────────
# Part B: BFS traversal layers — hop-by-hop discovery
# ─────────────────────────────────────────────────────────────────────────────
print()
print("== BFS traversal layers from 'lehman brothers' ==")

def bfs_layers(G: nx.DiGraph, seed: str, max_hops: int = 2) -> dict[int, list[str]]:
    '''Return nodes grouped by hop distance from seed.'''
    seed_n = normalise(seed)
    if seed_n not in G:
        return {}
    visited = {seed_n: 0}
    queue: deque[tuple[str, int]] = deque([(seed_n, 0)])
    layers: dict[int, list[str]] = defaultdict(list)
    layers[0].append(seed_n)
    while queue:
        node, depth = queue.popleft()
        if depth >= max_hops:
            continue
        for nbr in list(G.successors(node)) + list(G.predecessors(node)):
            if nbr not in visited:
                visited[nbr] = depth + 1
                layers[depth + 1].append(nbr)
                queue.append((nbr, depth + 1))
    return dict(layers)


layers = bfs_layers(KG, 'Lehman Brothers', max_hops=2)
for hop, nodes in sorted(layers.items()):
    label = 'SEED' if hop == 0 else f'HOP {hop}'
    print(f'  {label}: {', '.join(sorted(nodes))}')

# ─────────────────────────────────────────────────────────────────────────────
# Part C: Relationship type distribution
# ─────────────────────────────────────────────────────────────────────────────
print()
print('== Relationship type distribution ==')
pred_counts: dict[str, int] = defaultdict(int)
for _, _, data in KG.edges(data=True):
    for pred in data['predicates']:
        pred_counts[pred] += 1
for pred, count in sorted(pred_counts.items(), key=lambda x: x[1], reverse=True):
    bar = '#' * count
    print(f'  {pred:<30} {bar} ({count})')

# ─────────────────────────────────────────────────────────────────────────────
# Part D: Vector-only baseline vs graph-augmented
# ─────────────────────────────────────────────────────────────────────────────
print()
print('== Baseline comparison: vector-only vs graph-augmented ==')

vector_only_docs = VECTOR_INDEX.similarity_search_with_score(QUERY, k=TOP_K_VECTOR)
vector_only_context = '\n\n---\n\n'.join(
    f"[{doc.metadata.get('id', '?')}] {doc.page_content}"
    for doc, _ in vector_only_docs
)

baseline_response = client.messages.create(
    model=SONNET,
    max_tokens=400,
    system=(
        'You are a financial analyst. Answer using ONLY the provided passages. '
        'Do not invent facts.'
    ),
    messages=[{
        'role': 'user',
        'content': f'Question: {QUERY}\n\nPassages:\n{vector_only_context}',
    }],
)
baseline_answer = baseline_response.content[0].text

print()
print('[VECTOR-ONLY ANSWER]')
print('-' * 50)
print(baseline_answer[:400] + ('...' if len(baseline_answer) > 400 else ''))

print()
print('[GRAPH-AUGMENTED ANSWER (from Cell 4)]')
print('-' * 50)
print(answer[:400] + ('...' if len(answer) > 400 else ''))

print()
print('Key difference:')
print('  Vector-only  : retrieves passages that MENTION Lehman Brothers')
print('  Graph-augmented: additionally surfaces entities CONNECTED to Lehman,')
print('  their relationship types, and the 2-hop downstream exposure network --')
print('  structural information that is invisible to embedding similarity.')


## Cell 6: Fintech — Counterparty Network Risk Analysis
**What this demonstrates**: Graph RAG applied to a counterparty credit risk scenario.
The graph surfaces the exposure chain from a distressed institution through connected
counterparties, with vector search providing policy and regulatory context.
**Expected output**: Full exposure network map; structured risk tier analysis; regulatory obligations.

In [ ]:
# Counterparty network risk analysis — the canonical fintech Graph RAG use case.
# The query asks about systemic exposure across all connected entities,
# which requires both graph traversal (who is connected) and vector search
# (what do the policy and regulatory documents say about those exposures).

COUNTERPARTY_QUERY = (
    'Map the full counterparty exposure network for AIG Financial Products. '
    'Identify all institutions with direct or indirect exposure, the instruments '
    'involved, and what regulatory obligations were triggered by these exposures.'
)

print('=' * 70)
print('COUNTERPARTY NETWORK RISK ANALYSIS')
print('=' * 70)
print(f'\nQuery: {COUNTERPARTY_QUERY}\n')

# ── Retrieve ──────────────────────────────────────────────────────────────────
cp_retrieval = graph_rag_retrieve(COUNTERPARTY_QUERY, KG, VECTOR_INDEX, top_k=5)

print(f'Query entities     : {cp_retrieval["query_entities"]}')
print(f'Graph nodes reached: {cp_retrieval["graph_nodes_reached"]}')
print(f'Graph edges found  : {cp_retrieval["graph_edges_found"]}')
print()
print('--- Entity relationship map ---')
print(cp_retrieval['graph_context'])

# ── Synthesise with a risk-framed system prompt ───────────────────────────────
RISK_SYSTEM = (
    'You are a counterparty credit risk analyst. Using the entity relationship '
    'map and supporting passages, produce a structured risk assessment:\n'
    '1. DIRECT EXPOSURES: institutions with direct contractual links\n'
    '2. INDIRECT EXPOSURES: second-degree connections and contagion paths\n'
    '3. INSTRUMENTS: each instrument type with approximate notional where stated\n'
    '4. REGULATORY TRIGGERS: which rules or requirements were activated\n'
    '5. RISK ASSESSMENT: which connection represents highest systemic risk and why\n'
    'Ground every claim in the provided context. Use entity names exactly as they appear.'
)

cp_response = client.messages.create(
    model=SONNET,
    max_tokens=900,
    system=RISK_SYSTEM,
    messages=[{
        'role': 'user',
        'content': (
            f'Analyse: {COUNTERPARTY_QUERY}\n\n'
            f'Context:\n{cp_retrieval["merged_context"]}'
        ),
    }],
)

cp_answer = cp_response.content[0].text

print()
print('=' * 70)
print('COUNTERPARTY RISK ASSESSMENT')
print('=' * 70)
print(cp_answer)

# ── Why Graph RAG is essential for this query ─────────────────────────────────
print()
print('=' * 70)
print('WHY GRAPH RAG IS ESSENTIAL FOR THIS QUERY')
print('=' * 70)
print()
print('This query requires three types of information:')
print()
print('  1. ENTITY RELATIONSHIPS (graph-only)')
print('     The AIG -> Goldman Sachs CDS relationship is a graph edge assembled')
print('     from two sentence fragments across two different documents.')
print('     No individual chunk contains the full relationship.')
print()
print('  2. HOP-2 EXPOSURE PATHS (graph-only)')
print('     Goldman Sachs held CDS from AIG (hop 1).')
print('     Goldman Sachs also received TARP funding (hop 2 from AIG).')
print('     This contagion chain is only visible through two edge traversals.')
print()
print('  3. POLICY AND REGULATORY TEXT (vector-only)')
print('     Basel III CCR requirements and Dodd-Frank clearing mandates')
print('     are retrieved by vector similarity, not graph traversal.')
print()
print('Graph RAG combines both. Vector search alone answers:')
print('  "What documents mention AIG?"')
print('Graph RAG answers:')
print('  "What is AIG\'s full exposure network, through what instruments,')
print('   and what regulatory obligations apply to each connection?"')
